# 第12章 数据库应用开发

## 12.1 使用Python操作SQLite数据库

■ 已内嵌在Python中，使用时需要导入sqlite3。
■ 使用C语言开发，支持大多数SQL92标准，不支持外键限制。
■ 支持原子的、一致的、独立和持久的事务。
■ 通过数据库级上的独占性和共享锁定来实现独立事务，当多个线程和进程同一时间访问同一数据库时，只有一个可以写入数据。
■ 支持140TB的数据库，每个数据库完全存储在单个磁盘文件中，以B+数据结构的形式存储，一个数据库就是一个文件，通过复制即可实现备份。

## 12.1 使用Python操作SQLite数据库

■ 常用的SQLite可视化管理工具
* www.sqlabs.com ==> SQLiteManager
* SQLite Database Browser

| Name | Type | Schema |
|---|---|---|
| students | table | CREATE TABLE students (kecheng TEXT, xuehao TEXT, xingmi... |
| tiwen | table | CREATE TABLE tiwen (xuehao TEXT, shijian TEXT, defen NUM... |
| xueshengtiwen | table | CREATE TABLE xueshengtiwen (xuehao TEXT, wenti TEXT, shi... |
| tiku | table | CREATE TABLE tiku (kechengmingcheng TEXT, id INTEGER PRI... |
| tikufangwenqingkuang | table | CREATE TABLE tikufangwenqingkuang (xuehao TEXT, xingming... |
| kaoshi | table | CREATE TABLE kaoshi (shijian TEXT, shifouzhengque TEXT, ... |
| dianming | table | CREATE TABLE dianming (mac TEXT, ip TEXT, xuehao TEXT, s... |

## 12.1 使用Python操作SQLite数据库

■ 访问和操作SQLite数据时，需要首先导入sqlite3模块，然后创建一个与数据库关联的Connection对象：

In [ ]:
import sqlite3
# 导入模块
conn = sqlite3.connect('example.db') # 连接数据库

## 12.1 使用Python操作SQLite数据库

■ 成功创建Connection对象以后，再创建一个Cursor对象，并且调用Cursor对象的execute()方法来执行SQL语句创建数据表以及查询、插入、修改或删除数据库中的数据：

In [ ]:
c = conn.cursor()
# 创建表
c.execute('''CREATE TABLE IF NOT EXISTS stocks (date text, trans text, symbol text, qty real, price real)''')
# 插入一条记录
c.execute("INSERT INTO stocks VALUES ('2006-01-05', 'BUY', 'RHAT', 100, 35.14)")
# 提交当前事务,保存数据
conn.commit()
# 关闭数据库连接
conn.close()

## 12.1 使用Python操作SQLite数据库

■ 如果需要查询表中内容，那么重新创建Connection对象和Cursor对象之后，可以使用下面的代码来查询。

In [ ]:
conn = sqlite3.connect('example.db')
c = conn.cursor()
for row in c.execute('SELECT * FROM stocks ORDER BY price'):
    print(row)
conn.close()

## 12.1.1 Connection对象

| 方法 | 说明 |
|---|---|
| execute(sql[, parameters]) | 执行一条SQL语句 |
| executemany (sql[, parameters]) | 执行多条SQL语句 |
| cursor () | 返回连接的游标 |
| commit() | 提交当前事务，如果不提交的话，那么自上次调用commit()方法之后的所有修改都不会真正保存到数据库中 |
| rollback | 撤销当前事务，将数据库恢复至上次调用commit()方法后的状态 |
| close() | 关闭数据库连接 |
| create_function (name, num_params, func) | 创建可在SQL语句中调用的函数，其中name为函数名，num_params表示该函数可以接收的参数个数，func表示Python可调用对象 |

## 12.1.1 Connection对象

■ 在sqlite3连接中创建并调用自定义函数：

In [ ]:
import sqlite3
import hashlib

# 自定义Python函数
def md5sum(t):
    return hashlib.md5(t).hexdigest()

#在内存中创建临时数据库, 不需要提交事务
conn = sqlite3.connect(":memory:")

# 创建可在SQL调用的函数,其中第二个参数表示函数的参数个数
conn.create_function("md5", 1, md5sum)
cur = conn.cursor()

# 在SQL语句中调用自定义函数
cur.execute("select md5(?)", ["中国山东烟台".encode()])
print(cur.fetchone()[0])

## 12.1.2 Cursor对象

Cursor对象常用方法:
* `close(...)`: 关闭游标
* `execute(...)`: 执行SQL语句
* `executemany(...)`: 重复执行多次SQL语句
* `executescript(...)`: 一次执行多条SQL语句
* `fetchall(...)`: 从结果集中返回所有行记录
* `fetchmany(...)`: 从结果集中返回多行记录
* `fetchone(...)`: 从结果集中返回一行记录

## 12.1.2 Cursor对象

■ `execute(sql[, parameters])`: 该方法用于执行一条SQL语句，下面的代码演示了用法，以及为SQL语句传递参数的两种方法，分别使用问号和命名变量作为占位符。

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE people (name_last, age)")

who = "Dong"
age = 38

# 使用问号作为占位符
cur.execute("INSERT INTO people VALUES (?, ?)", (who, age))

# 使用命名变量作为占位符
cur.execute("SELECT * FROM people WHERE name_last=:who AND age=:age", {"who": who, "age": age})
print(cur.fetchone())

## 12.1.2 Cursor对象

■ `executemany(sql, seq_of_parameters)`: 该方法用来对于所有给定参数执行同一个SQL语句，参数序列可以使用不同的方式产生。

## 12.1.2 Cursor对象

√ 使用迭代来产生参数序列:

In [ ]:
import sqlite3

# 自定义迭代器,按顺序生成小写字母
class IterChars:
    def __init__(self):
        self.count = ord('a')
        
    def __iter__(self):
        return self
        
    def __next__(self):
        if self.count > ord('z'):
            raise StopIteration
        self.count += 1
        return (chr(self.count - 1),)

## 12.1.2 Cursor对象

In [ ]:
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE characters(c)")

# 创建迭代器对象
theIter = IterChars()

# 插入记录,每次插入一个英文小写字母
cur.executemany("INSERT INTO characters(c) VALUES (?)", theIter)

# 读取并显示所有记录
cur.execute("SELECT c FROM characters")
print(cur.fetchall())

## 12.1.2 Cursor对象

√ 使用生成器对象来产生参数:

In [ ]:
import sqlite3
import string

# 包含yield语句的函数可以用来创建生成器对象
def char_generator():
    for c in string.ascii_lowercase:
        yield (c,)

## 12.1.2 Cursor对象

In [ ]:
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE characters(c)")

# 使用生成器对象得到参数序列
cur.executemany("INSERT INTO characters (c) VALUES (?)", char_generator())

cur.execute("SELECT c FROM characters")
print(cur.fetchall())

## 12.1.2 Cursor对象

√ 使用直接创建的序列作为SQL语句的参数:

In [ ]:
import sqlite3
persons = [("Hugo", "Boss"), ("Calvin", "Klein")]
conn = sqlite3.connect(":memory:")

# 创建表
conn.execute("CREATE TABLE person(firstname, lastname)")

# 插入数据
conn.executemany("INSERT INTO person (firstname, lastname) VALUES (?, ?)", persons)

# 显示数据
for row in conn.execute("SELECT firstname, lastname FROM person"):
    print(row)
    
print("I just deleted", conn.execute("DELETE FROM person").rowcount, "rows")

## 12.1.2 Cursor对象

■ `fetchone()`、`fetchmany(size=cursor.arraysize)`、`fetchall()`: 用来读取数据。假设数据库通过下面代码插入数据:

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:") # 修改为内存数据库方便演示运行
cur = conn.cursor() # 创建游标

# 补充创建表的代码使得语句能运行
cur.execute('''CREATE TABLE addressList(name TEXT, sex TEXT, phon TEXT, QQ TEXT, address TEXT)''')

cur.execute('''INSERT INTO addressList(name, sex, phon, QQ, address) VALUES('王小丫', '女', '13888997011','66735','北京市')''')
cur.execute('''INSERT INTO addressList(name, sex, phon, QQ, address) VALUES('李莉', '女', '15808066055', '675797', '天津市')''')
cur.execute('''INSERT INTO addressList(name, sex, phon, QQ, address) VALUES('李星草', '男', '15912108090', '3232099', '昆明市')''')

conn.commit() # 提交事务,把数据写入数据库
# conn.close() # 暂不关闭以便后续查询

## 12.1.2 Cursor对象

√ 使用fetchall()读取数据:

In [ ]:
# 前接上一页运行状态，或者重新连接:
# import sqlite3
# conn = sqlite3.connect('D:/addressBook.db')
# cur = conn.cursor()

cur.execute('SELECT * FROM addressList')
li = cur.fetchall()

for line in li:
    for item in line:
        print(item, end=' ')
    print()
    
conn.close() # 返回所有查询结果

## 12.1.3 Row对象

■ 假设数据以下面的方式创建并插入数据:

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:") # 原文 D:\test.db
c = conn.cursor()
c.execute('''CREATE TABLE stocks (date text, trans text, symbol text, qty real, price real)''')
c.execute("""INSERT INTO stocks VALUES ('2006-01-05', 'BUY', 'RHAT', 100, 35.14)""")
conn.commit()

## 12.1.3 Row对象

√ 使用fetchone()方法读取其中数据:

In [ ]:
conn.row_factory = sqlite3.Row
c = conn.cursor()
c.execute('SELECT * FROM stocks')
r = c.fetchone()

print(type(r))
print(tuple(r))
print(r[2])
print(r.keys())
print(r['qty'])

for field in r:
    print(field)
    
c.close()

## 12.2 访问其他类型数据库

■ 除了SQLite数据库以外，Python还可以操作ACCESS、SQLServer、MYSQL等多种类型的数据库。

## 12.2.1 操作ACCESS数据库

■ 需要首先安装Python for Windows extensions, Pywin32。

√ 建立数据库连接

In [ ]:
''' # 注意：由于运行环境限制，以下涉及Windows特定组件的伪代码仅供展示格式
import win32com.client
conn = win32com.client.Dispatch(r'ADODB.Connection')
DSN = 'PROVIDER=Microsoft.Jet.OLEDB.4.0;DATA SOURCE=C:/MyDB.mdb;'
conn.Open(DSN)

# 打开记录集
rs = win32com.client.Dispatch(r'ADODB.Recordset')
rs_name = 'MyRecordset' # 表名
rs.Open('[' + rs_name + ']', conn, 1, 3)
'''

## 12.2.1 操作ACCESS数据库

√ 操作记录集

In [ ]:
'''
rs.AddNew()
rs.Fields.Item(1).Value = 'data'
rs.Update()
'''

## 12.2.1 操作ACCESS数据库

√ 操作数据

In [ ]:
'''
conn = win32com.client.Dispatch(r'ADODB.Connection')
DSN = 'PROVIDER=Microsoft.Jet.OLEDB.4.0; DATA SOURCE=C:/MyDB.mdb;'
sql_statement = "Insert INTO [Table_Name] ([Field_1], [Field_2]) VALUES ('data1', 'data2')"
conn.Open(DSN)
conn.Execute(sql_statement)
conn.Close()
'''

## 12.2.1 操作ACCESS数据库

√ 遍历记录

In [ ]:
'''
rs.MoveFirst()
count = 0
while 1:
    if rs.EOF:
        break
    else:
        count = count + 1
        rs.MoveNext()
'''

## 12.2.2 操作MSSQLServer数据库

■ 使用Pywin32，需要首先安装

√ 导入模块:

In [ ]:
'''
import adodbapi
adodbapi.adodbapi.verbose = False # adds details to the sample printout
import adodbapi.ado_consts as adc
'''

## 12.2.2 操作MSSQLServer数据库

√ 创建连接:

In [ ]:
'''
Cfg={'server':'192.168.29.86\\eclexpress','password':'xxxx','db':'pscitemp'}
constr = r"Provider=SQLOLEDB.1; Initial Catalog=%s; Data Source=%s; user ID=%s; Password=%s; " % (Cfg['db'], Cfg['server'], 'sa', Cfg['password'])
conn=adodbapi.connect(constr)
'''

## 12.2.2 操作MSSQLServer数据库

√ 执行sql语句:

In [ ]:
'''
cur=conn.cursor()
sql='''select * from softextBook where title='{0}' and remark3!='{1}''''.format(bookName, flag)
cur.execute(sql)
data=cur.fetchall()
cur.close()
'''

## 12.2.2 操作MSSQLServer数据库

√ 执行存储过程:

In [ ]:
'''
#假设proName有三个参数,最后一个参数传了null
ret=cur.callproc('procName', (parm1, parm2, None))
conn.commit()

# 关闭连接
conn.close()
'''

## 12.2.2 操作MSSQLServer数据库

■ 使用pymssql，需要首先安装

In [ ]:
'''
import pymssql
conn = pymssql.connect(host='SQL01', user='user', password='password', database='mydatabase')
cur = conn.cursor()

cur.execute('CREATE TABLE persons (id INT, name VARCHAR(100))')
cur.executemany("INSERT INTO persons VALUES(%d, %s)", [(1, 'John Doe'), (2, 'Jane Doe')])
conn.commit()

cur.execute('SELECT * FROM persons WHERE name=%s', 'John Doe')
row = cur.fetchone()
while row:
    print("ID=%d, Name=%s" % (row[0], row[1]))
    row = cur.fetchone()
    
cur.execute("SELECT * FROM persons WHERE name LIKE 'J%'")
conn.close()
'''

## 12.2.3 操作MySQL数据库

■ 首先需要安装MySQLDb
http://trac.edgewall.org/wiki/MySqlDb

## 12.2.3 操作MySQL数据库

■ MySQLDb模块的主要方法:
✓ `commit()`: 提交事务。
✓ `rollback()`: 回滚事务。
✓ `callproc(self, procname, args)`: 用来执行存储过程，接收的参数为存储过程名和参数列表，返回值为受影响的行数。
✓ `execute(self, query, args)`: 执行单条sql语句，接收的参数为sql语句本身和使用的参数列表，返回值为受影响的行数。
✓ `executemany(self, query, args)`: 执行单条sql语句，但是重复执行参数列表里的参数，返回值为受影响的行数。

## 12.2.3 操作MySQL数据库

* `nextset(self)`: 移动到下一个结果集。
* `fetchall(self)`: 接收全部的返回结果行。
* `fetchmany(self, size=None)`: 接收size条返回结果行，如果size的值大于返回的结果行的数量，则会返回cursor.arraysize条数据。
* `fetchone(self)`: 返回一条结果行。
* `scroll(self, value, mode='relative')`: 移动指针到某一行。如果mode='relative'则表示从当前所在行移动value条，如果mode='absolute'则表示从结果集的第一行移动value条。

## 12.2.3 操作MySQL数据库

■ 查询记录

In [ ]:
'''
import MySQLdb
try:
    conn=MySQLdb.connect(host='localhost',user='root', passwd='root',db='test', port=3306)
    cur=conn.cursor()
    cur.execute('select * from user')
    cur.close()
    conn.close()
except MySQLdb.Error as e:
    print("Mysql Error %d: %s" % (e.args[0], e.args[1]))
'''

## 12.2.3 操作MySQL数据库

■ 插入数据

In [ ]:
'''
import MySQLdb
try:
    conn=MySQLdb.connect(host='localhost',user='root', passwd='root', port=3306)
    cur=conn.cursor()
    cur.execute('create database if not exists python')
    conn.select_db('python')
    cur.execute('create table test(id int, info varchar(20))')
    
    value = [1, 'hi rollen']
    cur.execute('insert into test values(%s,%s)', value)
    
    values = []
    for i in range(20):
        values.append((i, 'hi rollen'+str(i)))
    
    cur.executemany('insert into test values(%s,%s)', values)
    cur.execute('update test set info="I am rollen" where id=3')
    
    conn.commit()
    cur.close()
    conn.close()
except MySQLdb.Error as e:
    print("Mysql Error %d: %s" % (e.args[0], e.args[1]))
'''

## 12.3 操作MongoDb数据库

■ MongoDB是一个基于分布式文件存储的文档数据库，可以说是非关系型(NoSQL, Not Only SQL)数据库中比较像关系型数据库的一个，具有免费、操作简单、面向文档存储、自动分片、可扩展性强、查询功能强大等特点，对大数据处理支持较好，旨在为WEB应用提供可扩展的高性能数据存储解决方案。
■ MongoDB将数据存储为一个文档，数据结构由键值(key=>value)对组成。MongoDB文档类似于JSON对象。字段值可以包含其他文档，数组及文档数组。

## 12.3 操作MongoDb数据库

■ MongoDB数据库可以到网站https://www.mongodb.org/downloads下载，安装之后打开命令提示符环境并切换到MongoDB安装目录中的server\3.2\bin文件夹，然后执行命令mongod --dbpath D:\data --journal --storageEngine=mmapv1 启动MongoDB，当然需要首先在D盘根目录下新建文件夹data。
■ 让刚才那个命令提示符环境始终处于运行状态，然后再打开一个命令提示符环境，执行mongo命令连接MongoDB数据库，如果连接成功的话，会显示一个>符号作为提示符，之后就可以输入MongoDB命令了

## 12.3 操作MongoDb数据库

■ 打开或创建collection
`> use students`

■ 在collection中插入数据
`> zhangsan = {'name': 'Zhangsan', 'age':18, 'sex':'male'}`
`> db.students.insert(zhangsan)`
`> lisi = {'name':'Lisi', 'age':19, 'sex':'male'}`
`> db.students.insert(lisi)`

■ 查询collection中的记录
`> db.students.find()`

■ 查看系统中所有数据库名称
`> show dbs`

## 12.3 操作MongoDb数据库

■ Python扩展库pymongo完美支持MongoDB数据的操作

In [ ]:
''' # 此处依赖MongoDB环境，用伪代码包裹以便讲义展示
import pymongo

#连接数据库,27017是默认端口
client = pymongo.MongoClient('localhost', 27017)

#获取数据库
db = client.students

#查看数据集合名称列表
print(db.collection_names())

#获取数据集合
students = db.students
print(students.find())

#遍历数据
for item in students.find():
    print(item)
'''

## 12.3 操作MongoDb数据库

In [ ]:
'''
wangwu = {'name': 'Wangwu', 'age':20, 'sex':'male'}

# 插入一条记录
print(students.insert(wangwu))

# 指定查询条件
for item in students.find({'name':'Wangwu'}):
    print(item)
    
# 获取一条记录
print(students.find_one())
print(students.find_one({'name': 'Wangwu'}))

# 记录总数
print(students.find().count())
'''

## 12.3 操作MongoDb数据库

In [ ]:
'''
# 删除一条记录
print(students.remove({'name': 'Wangwu'}))

for item in students.find():
    print(item)
    
print(students.find().count())

# 创建索引
print(students.create_index([('name', pymongo.ASCENDING)]))

# 更新数据库
print(students.update({'name':'Zhangsan'},{'$set':{'age':25}}))

# 更新数据库
print(students.update({'age':25},{'$set':{'sex':'Female'}}))

# 清空数据库
print(students.remove())
'''

## 12.3 操作MongoDb数据库

In [ ]:
'''
Zhangsan = {'name': 'Zhangsan', 'age':20, 'sex':'Male'}
Lisi = {'name': 'Lisi', 'age':21, 'sex':'Male'}
Wangwu = {'name': 'Wangwu', 'age':22, 'sex':'Female'}

# 插入多条数据
print(students.insert_many([Zhangsan, Lisi, Wangwu]))

# 对查询结果排序
for item in students.find().sort('name',pymongo.ASCENDING):
    print(item)
    
for item in students.find().sort([('sex',pymongo.DESCENDING), ('name', pymongo.ASCENDING)]):
    print(item)
'''